In [16]:
import os
import pandas as pd
from dotenv import load_dotenv


# 載入環境變數中的 Token
load_dotenv()
token = os.getenv("FINMIND_TOKEN")

def get_financial_data(stock_id, start_date):
    """
    具備 Cache 機制的資料抓取函數
    邏輯：優先檢查 data/ 資料夾，若無才呼叫 API
    """
    # 建立存檔路徑 (例如: data/2330_financial_statement.csv)
    file_path = f"data/{stock_id}_financial.csv"
    

    # --- 1. 檢查快取 (DSA 邏輯：減少重複運算/請求) ---
    if os.path.exists(file_path):
        print(f"找到本地快取：{file_path}，直接讀取...")
        return pd.read_csv(file_path)

    # --- 2. 呼叫 API ---
    print(f"正在從 FinMind 抓取 {stock_id} 的財報資料...")
    dl = DataLoader()

    
    # 抓取綜合損益表
    df = dl.taiwan_stock_financial_statement(
        stock_id=stock_id,
        start_date=start_date
    )

    # --- 3. 儲存快取 (Persistence) ---
    if not df.empty:
        df.to_csv(file_path, index=False)
        print(f"資料抓取成功並已存至 {file_path}")
    else:
        print("警告：API 回傳空資料，請檢查 Token 或股票代碼。")
        
    return df

# --- 執行測試 ---
# 試試看抓取台積電 (2330)
df_2330 = get_financial_data("2330", "2023-01-01")
df_2330.head()


找到本地快取：data/2330_financial.csv，直接讀取...


,date,stock_id,type,value,origin_name
0,2023-03-31,2330,OperatingExpenses,5.530934e+10,營業費用
1,2023-03-31,2330,TAX,3.732590e+10,所得稅費用（利益）
2,2023-03-31,2330,EPS,7.980000e+00,基本每股盈餘（元）
3,2023-03-31,2330,PreTaxIncome,2.442749e+11,稅前淨利（淨損）
4,2023-03-31,2330,IncomeFromContinuingOperations,2.069490e+11,繼續營業單位本期淨利（淨損）


In [2]:
# =============================================================================
# CacheManager：快取管理器
# =============================================================================
# 功能：檢查、讀取、寫入快取，支援不同資料類型的過期策略
# 對標 C++：類似封裝好的 std::map + std::filesystem 操作
# =============================================================================

import pandas as pd
from pathlib import Path
from datetime import datetime

class CacheManager:
    """
    快取管理器
    
    職責：
    1. 生成快取檔案路徑 (依據 stock_id, data_type, 日期範圍)
    2. 檢查快取是否存在且有效 (未過期)
    3. 讀取/寫入快取
    """
    
    # --- 類別常數：各資料類型的過期天數 ---
    # 對標 C++：static const std::map<std::string, int>
    EXPIRY_DAYS = {
        "stock_price": 1,           # 日股價：FinMind 每天 17:30 更新
        "financial_statement": 90,  # 財報：每季更新一次
        "monthly_revenue": 30,      # 月營收：每月 10 號前公布
    }
    
    def __init__(self, cache_dir: str = "data"):
        """
        建構子
        
        Args:
            cache_dir: 快取根目錄，預設為 "data"
        
        對標 C++：
            CacheManager(const std::string& cache_dir = "data") 
                : cache_dir_(cache_dir) {
                std::filesystem::create_directories(cache_dir_);
            }
        """
        # Path 是路徑物件，支援 / 運算子串接路徑
        # 對標 C++：std::filesystem::path
        self.cache_dir = Path(cache_dir)
        
        # 建立快取目錄（如果不存在）
        # parents=True：自動建立父目錄
        # exist_ok=True：目錄已存在不報錯
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    
    def _generate_key(self, stock_id: str, data_type: str, 
                      start_date: str, end_date: str) -> Path:
        """
        生成快取檔案路徑（私有方法）
        
        Args:
            stock_id: 股票代號，如 "2330"
            data_type: 資料類型，如 "financial_statement"
            start_date: 起始日期，如 "2023-01-01"
            end_date: 結束日期，如 "2024-01-01"
        
        Returns:
            完整檔案路徑，如 data/financial_statement/2330_2023-01-01_2024-01-01.csv
        
        為什麼 key 要包含所有參數？
            不同查詢條件 = 不同資料內容 = 不同快取檔案
            避免 start_date 不同但讀到同一個快取的錯誤
        """
        # 建立子資料夾（依資料類型分類）
        # self.cache_dir / data_type 等於 "data" / "financial_statement"
        type_dir = self.cache_dir / data_type
        type_dir.mkdir(exist_ok=True)
        
        # 組合檔名：股票代號_起始日期_結束日期.csv
        filename = f"{stock_id}_{start_date}_{end_date}.csv"
        
        # 回傳完整路徑
        return type_dir / filename
    
    def _is_expired(self, file_path: Path, data_type: str) -> bool:
        """
        檢查快取是否過期（私有方法）
        
        Args:
            file_path: 快取檔案路徑
            data_type: 資料類型（用來查詢過期天數）
        
        Returns:
            True = 已過期，False = 未過期
        
        對標 C++：
            auto mtime = std::filesystem::last_write_time(path);
            auto now = std::chrono::system_clock::now();
            return (now - mtime) > expiry_days * 24h;
        """
        # stat() 取得檔案狀態，st_mtime 是最後修改時間（Unix timestamp）
        # fromtimestamp() 把 timestamp 轉成 datetime 物件
        mtime = datetime.fromtimestamp(file_path.stat().st_mtime)
        
        # 計算距今幾天
        days_passed = (datetime.now() - mtime).days
        
        # 查詢該資料類型的過期天數，預設 1 天
        # dict.get(key, default)：找不到 key 時回傳 default
        expiry = self.EXPIRY_DAYS.get(data_type, 1)
        
        return days_passed > expiry
    
    def get(self, stock_id: str, data_type: str,
            start_date: str, end_date: str) -> pd.DataFrame | None:
        """
        讀取快取
        
        Args:
            stock_id: 股票代號
            data_type: 資料類型
            start_date: 起始日期
            end_date: 結束日期
        
        Returns:
            DataFrame：快取存在且未過期
            None：快取不存在或已過期
        
        對標 C++：
            std::optional<DataFrame> get(...) {
                if (!exists || expired) return std::nullopt;
                return loadCSV(path);
            }
        """
        file_path = self._generate_key(stock_id, data_type, start_date, end_date)
        
        # 檢查 1：檔案是否存在
        if not file_path.exists():
            print(f"[Cache] 快取不存在：{file_path}")
            return None
        
        # 檢查 2：是否過期
        if self._is_expired(file_path, data_type):
            print(f"[Cache] 快取已過期：{file_path}")
            return None
        
        # 快取命中，讀取並回傳
        print(f"[Cache] 命中快取：{file_path}")
        return pd.read_csv(file_path)
    
    def set(self, stock_id: str, data_type: str,
            start_date: str, end_date: str, df: pd.DataFrame) -> None:
        """
        寫入快取
        
        Args:
            stock_id: 股票代號
            data_type: 資料類型
            start_date: 起始日期
            end_date: 結束日期
            df: 要儲存的 DataFrame
        """
        file_path = self._generate_key(stock_id, data_type, start_date, end_date)
        
        # 儲存為 CSV
        # index=False：不儲存 pandas 自動生成的列索引（0, 1, 2...）
        df.to_csv(file_path, index=False)
        print(f"[Cache] 已寫入快取：{file_path}")


# --- 測試 CacheManager ---
cache = CacheManager("data")
print("CacheManager 初始化成功")
print(f"快取目錄：{cache.cache_dir}")
print(f"過期規則：{cache.EXPIRY_DAYS}")

CacheManager 初始化成功
快取目錄：data
過期規則：{'stock_price': 1, 'financial_statement': 90, 'monthly_revenue': 30}


In [4]:
# =============================================================================
# DataFetcher：資料獲取器
# =============================================================================
# 功能：負責呼叫 FinMind API 獲取各類股票資料
# 職責單一：只管 API 呼叫，不管快取（快取由 CacheManager 負責）
# =============================================================================

from FinMind.data import DataLoader
import pandas as pd
import os

class DataFetcher:
    """
    資料獲取器
    斯41
    職責：
    1. 管理 FinMind API Token
    2. 呼叫各種 FinMind API
    3. 錯誤處理（API 失敗時回傳空 DataFrame）
    
    對標 C++：
    - 類似一個封裝好的 HTTP Client
    - 每個 public method 對應一種 API endpoint
    """
    
    def __init__(self, token: str = None):
        """
        建構子
        
        Args:
            token: FinMind API Token，若不傳則從環境變數讀取
        
        對標 C++：
            DataFetcher(const std::string& token = "") {
                if (token.empty()) {
                    token_ = std::getenv("FINMIND_TOKEN");
                }
            }
        """
        # 如果沒傳 token，從環境變數讀取
        # 這樣 token 不會寫死在程式碼裡（安全性考量）
        self.token = token or os.getenv("FINMIND_TOKEN")
        
        # 初始化 FinMind DataLoader
        self.loader = DataLoader()
        
        # 如果有 token，設定給 loader（提高 API 額度）
        if self.token:
            self.loader.login_by_token(api_token=self.token)
            print("[DataFetcher] 已使用 Token 登入（600次/小時）")
        else:
            print("[DataFetcher] 未設定 Token，使用匿名模式（300次/小時）")
    
    def get_financial_statement(self, stock_id: str, 
                                 start_date: str, end_date: str) -> pd.DataFrame:
        """
        獲取財務報表（綜合損益表）
        
        Args:
            stock_id: 股票代號，如 "2330"
            start_date: 起始日期，如 "2023-01-01"
            end_date: 結束日期，如 "2024-01-01"
        
        Returns:
            DataFrame：成功時回傳財報資料
            空 DataFrame：失敗時回傳（不是 None，方便後續用 .empty 檢查）
        
        為什麼用 try-except？
            API 可能因為網路問題、Token 過期、股票代號錯誤等原因失敗
            用 try-except 確保程式不會崩潰
        
        對標 C++：
            try { ... } catch (const std::exception& e) { ... }
        """
        try:
            print(f"[DataFetcher] 正在獲取 {stock_id} 的財報資料...")
            
            # 呼叫 FinMind API
            df = self.loader.taiwan_stock_financial_statement(
                stock_id=stock_id,
                start_date=start_date,
                end_date=end_date
            )
            
            # 檢查是否有資料
            if df.empty:
                print(f"[DataFetcher] 警告：{stock_id} 財報查無資料")
            else:
                print(f"[DataFetcher] 成功獲取 {len(df)} 筆財報資料")
            
            return df
            
        except Exception as e:
            # 印出錯誤訊息，但不讓程式崩潰
            print(f"[DataFetcher] 錯誤：獲取財報失敗 - {e}")
            # 回傳空 DataFrame（不是 None）
            return pd.DataFrame()
    
    def get_stock_price(self, stock_id: str, 
                        start_date: str, end_date: str) -> pd.DataFrame:
        """
        獲取股價資料（日K線）
        
        Args:
            stock_id: 股票代號
            start_date: 起始日期
            end_date: 結束日期
        
        Returns:
            DataFrame：股價資料（日期、開高低收、成交量等）
        """
        try:
            print(f"[DataFetcher] 正在獲取 {stock_id} 的股價資料...")
            
            df = self.loader.taiwan_stock_daily(
                stock_id=stock_id,
                start_date=start_date,
                end_date=end_date
            )
            
            if df.empty:
                print(f"[DataFetcher] 警告：{stock_id} 股價查無資料")
            else:
                print(f"[DataFetcher] 成功獲取 {len(df)} 筆股價資料")
            
            return df
            
        except Exception as e:
            print(f"[DataFetcher] 錯誤：獲取股價失敗 - {e}")
            return pd.DataFrame()
    
    def get_monthly_revenue(self, stock_id: str, 
                            start_date: str, end_date: str) -> pd.DataFrame:
        """
        獲取月營收資料
        
        Args:
            stock_id: 股票代號
            start_date: 起始日期
            end_date: 結束日期
        
        Returns:
            DataFrame：月營收資料
        """
        try:
            print(f"[DataFetcher] 正在獲取 {stock_id} 的月營收資料...")
            
            df = self.loader.taiwan_stock_month_revenue(
                stock_id=stock_id,
                start_date=start_date,
                end_date=end_date
            )
            
            if df.empty:
                print(f"[DataFetcher] 警告：{stock_id} 月營收查無資料")
            else:
                print(f"[DataFetcher] 成功獲取 {len(df)} 筆月營收資料")
            
            return df
            
        except Exception as e:
            print(f"[DataFetcher] 錯誤：獲取月營收失敗 - {e}")
            return pd.DataFrame()


# --- 測試 DataFetcher ---
fetcher = DataFetcher()
print(f"Token 狀態：{'已設定' if fetcher.token else '未設定'}")

2026-01-20 20:23:11.465 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-01-20 20:23:11.547 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


[DataFetcher] 已使用 Token 登入（600次/小時）
Token 狀態：已設定


In [ ]:
# =============================================================================
# DataService：資料服務層（整合 CacheManager + DataFetcher）
# =============================================================================
# 功能：統一的資料獲取介面，自動處理快取邏輯
# 設計模式：Facade Pattern（外觀模式）- 把複雜的子系統包裝成簡單的介面
# =============================================================================

class DataService:
    """
    資料服務層
    
    職責：
    1. 整合 CacheManager 和 DataFetcher
    2. 提供統一的 get_data() 介面
    3. 自動處理「先查快取 → 沒有才打 API → 存入快取」的流程
    
    對標 C++：
    - Facade Pattern：把多個 class 的操作包裝成單一介面
    - 類似 Service Layer 的概念
    """
    
    def __init__(self, cache_dir: str = "data", token: str = None):
        """
        建構子
        
        Args:
            cache_dir: 快取目錄
            token: FinMind API Token
        """
        # 初始化子系統
        self.cache = CacheManager(cache_dir)
        self.fetcher = DataFetcher(token)
        
        # 建立「資料類型 → API 方法」的對應表
        # dict 的 value 是函式本身（不是呼叫結果）
        # 對標 C++：std::map<std::string, std::function<...>>
        self._fetch_methods = {
            "financial_statement": self.fetcher.get_financial_statement,
            "stock_price": self.fetcher.get_stock_price,
            "monthly_revenue": self.fetcher.get_monthly_revenue,
        }
    
    def get_data(self, stock_id: str, data_type: str,
                 start_date: str, end_date: str) -> pd.DataFrame:
        """
        統一的資料獲取介面
        
        流程：
        1. 查快取 → 有且未過期 → 直接回傳
        2. 快取沒有或過期 → 打 API
        3. API 成功 → 存入快取 → 回傳
        4. API 失敗 → 回傳空 DataFrame
        
        Args:
            stock_id: 股票代號，如 "2330"
            data_type: 資料類型，如 "financial_statement", "stock_price", "monthly_revenue"
            start_date: 起始日期，如 "2023-01-01"
            end_date: 結束日期，如 "2024-01-01"
        
        Returns:
            DataFrame：股票資料
        
        使用範例：
            service = DataService()
            df = service.get_data("2330", "financial_statement", "2023-01-01", "2024-01-01")
        """
        # --- Step 1: 查快取 ---
        df = self.cache.get(stock_id, data_type, start_date, end_date)
        
        if df is not None:
            # 快取命中，直接回傳
            return df
        
        # --- Step 2: 快取沒有，打 API ---
        # 檢查 data_type 是否有效
        if data_type not in self._fetch_methods:
            print(f"[DataService] 錯誤：不支援的資料類型 '{data_type}'")
            print(f"[DataService] 支援的類型：{list(self._fetch_methods.keys())}")
            return pd.DataFrame()
        
        # 從 dict 取出對應的方法並呼叫
        # self._fetch_methods[data_type] 是一個函式
        # 後面的 (...) 是呼叫這個函式
        fetch_method = self._fetch_methods[data_type]
        df = fetch_method(stock_id, start_date, end_date)
        
        # --- Step 3: 存入快取 ---
        if not df.empty:
            self.cache.set(stock_id, data_type, start_date, end_date, df)
        
        return df
    
    def get_supported_types(self) -> list:
        """
        取得支援的資料類型列表
        
        Returns:
            list：支援的資料類型，如 ["financial_statement", "stock_price", "monthly_revenue"]
        """
        return list(self._fetch_methods.keys())


# --- 測試 DataService ---
service = DataService()
print("DataService 初始化成功")
print(f"支援的資料類型：{service.get_supported_types()}")

In [ ]:
def transform_to_sentences(df):
    """
    將 DataFrame 的數字轉化為 AI 好讀的自然語言
    """
    sentences = []
    # 篩選我們感興趣的指標 (例如：營收、淨利)
    target_indicators = ['Revenue', 'Net_Income_Loss', 'EPS']
    filtered_df = df[df['type'].isin(target_indicators)]

    for _, row in filtered_df.iterrows():
        sentence = f"{row['date']} {row['stock_id']} 的 {row['type']} 為 {row['value']} 元。"
        sentences.append(sentence)
    
    return sentences

# 轉化並列印前五條
report_sentences = transform_to_sentences(df_2330)
for s in report_sentences[:5]:
    print(f"生成文本: {s}")